# 🛍️ Shopping Mall Customer Segmentation
## Member 2 — MeanShift Clustering
**Dataset:** Shopping_Mall_Customer_Segmentation_Data_.csv  
**Algorithm:** MeanShift Clustering  
**Task:** Unsupervised Machine Learning — Customer Segmentation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Load Pre-processed Data
> Data was already cleaned and scaled in `step0_eda_preprocessing.ipynb`.


In [ ]:
# Load preprocessed data from step0_eda_preprocessing.ipynb output
# ⚠️ Make sure you have run step0_eda_preprocessing.ipynb first!
import numpy as np
import pandas as pd

X_scaled = np.load('X_scaled.npy')
df_clean  = pd.read_csv('data/data_preprocessed.csv')
df        = pd.read_csv('data/Shopping Mall Customer Segmentation Data .csv')

print('✅ Loaded X_scaled.npy   shape:', X_scaled.shape)
print('✅ Loaded data_preprocessed.csv shape:', df_clean.shape)
print('✅ Loaded original CSV   shape:', df.shape)

## 2b. Re-fit Scaler
> Needed for transforming new customer input during prediction.


In [ ]:
# Re-fit scaler on loaded data (needed for prediction inverse transform)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(df_clean)
print('✅ Scaler re-fitted on preprocessed data')

## 5. Bandwidth Estimation
> MeanShift does NOT require you to specify the number of clusters. Instead, it uses a **bandwidth** parameter to determine cluster density. We use `estimate_bandwidth` to find a good value automatically.

In [ ]:
# Use a sample for speed on large datasets
sample_size = min(3000, len(X_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_idx]

bandwidth = estimate_bandwidth(X_sample, quantile=0.15, n_samples=500, random_state=42)
print(f'Estimated Bandwidth: {bandwidth:.4f}')

# Try a few bandwidth values
print('\nBandwidth sensitivity check:')
for q in [0.15, 0.20, 0.25, 0.30]:
    bw = estimate_bandwidth(X_sample, quantile=q, n_samples=500, random_state=42)
    ms_test = MeanShift(bandwidth=bw, bin_seeding=True)
    ms_test.fit(X_sample)
    n_clust = len(np.unique(ms_test.labels_))
    print(f'  quantile={q} → bandwidth={bw:.4f} → clusters found={n_clust}')

## 6. Train MeanShift Model

In [ ]:
# Train on full dataset with chosen bandwidth
# Increase bandwidth if too many clusters; decrease for more granularity
ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms.fit(X_scaled)

cluster_labels = ms.labels_
n_clusters = len(np.unique(cluster_labels))

print(f'Number of clusters found by MeanShift: {n_clusters}')
print('\nCluster Distribution:')
unique, counts = np.unique(cluster_labels, return_counts=True)
for c, cnt in zip(unique, counts):
    print(f'  Cluster {c}: {cnt} customers ({cnt/len(df)*100:.1f}%)')

df_clean['Cluster'] = cluster_labels
df['Cluster'] = cluster_labels

## 7. Evaluation Metrics

In [ ]:
sil = silhouette_score(X_scaled, cluster_labels)
dbi = davies_bouldin_score(X_scaled, cluster_labels)

print('=' * 45)
print('     MEANSHIFT EVALUATION METRICS')
print('=' * 45)
print(f'  Bandwidth Used      : {bandwidth:.4f}')
print(f'  Number of Clusters  : {n_clusters}')
print(f'  Silhouette Score    : {sil:.4f}  (closer to 1 = better)')
print(f'  Davies-Bouldin Index: {dbi:.4f}  (closer to 0 = better)')
print('=' * 45)

## 8. Cluster Visualization (PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained Variance by 2 PCA components: {pca.explained_variance_ratio_.sum()*100:.2f}%')

plt.figure(figsize=(9, 6))
palette = sns.color_palette('tab10', n_clusters)
for c in range(n_clusters):
    mask = cluster_labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                color=palette[c], label=f'Cluster {c}', alpha=0.5, s=15)

# Plot cluster centers
centers_pca = pca.transform(ms.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            marker='*', s=300, c='black', zorder=5, label='Cluster Centers')

plt.title(f'MeanShift Clustering ({n_clusters} clusters) — PCA 2D View')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.savefig('meanshift_pca_clusters.png', dpi=150)
plt.show()

## 9. Cluster Profile Analysis

In [ ]:
profile = df.groupby('Cluster')[['Age', 'Annual Income', 'Spending Score']].mean().round(2)
profile['Count'] = df.groupby('Cluster')['Age'].count()
profile['% of Total'] = (profile['Count'] / len(df) * 100).round(2)
print('Cluster Profiles:')
profile

In [ ]:
# Bar chart of cluster averages
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
features = ['Age', 'Annual Income', 'Spending Score']
palette_bar = sns.color_palette('tab10', n_clusters)

for i, feat in enumerate(features):
    cluster_means = df.groupby('Cluster')[feat].mean()
    axes[i].bar(cluster_means.index.astype(str), cluster_means.values, color=palette_bar)
    axes[i].set_title(f'Average {feat} per Cluster')
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel(feat)

plt.tight_layout()
plt.savefig('meanshift_cluster_profiles.png', dpi=150)
plt.show()

## 10. Summary

In [ ]:
print('=' * 55)
print('       MEANSHIFT CLUSTERING — FINAL SUMMARY')
print('=' * 55)
print(f'  Algorithm             : MeanShift')
print(f'  Bandwidth Used        : {bandwidth:.4f}')
print(f'  Clusters Auto-Found   : {n_clusters} (no K needed!)')
print(f'  Total Customers       : {len(df)}')
print(f'  Silhouette Score      : {sil:.4f}')
print(f'  Davies-Bouldin Index  : {dbi:.4f}')
print('=' * 55)
print('\nKey advantage of MeanShift:')
print('  - Does NOT require specifying K in advance')
print('  - Automatically finds number of clusters based on data density')

---
# 🔮 Predict Customer Segment
> Enter a new customer's details below and the trained model will predict which segment they belong to.

In [ ]:
# ============================================================
#  ✏️  ENTER CUSTOMER DETAILS HERE
# ============================================================
input_age           = int(input('Enter Age           : '))
input_gender        = input('Enter Gender        (Male/Female): ').strip().capitalize()
input_annual_income = float(input('Enter Annual Income : '))
input_spending_score= float(input('Enter Spending Score (1-100): '))

gender_encoded = 0 if input_gender == 'Female' else 1

new_customer = np.array([[input_age, gender_encoded, input_annual_income, input_spending_score]])
new_customer_scaled = scaler.transform(new_customer)

# MeanShift prediction — finds nearest cluster center
predicted_cluster = ms.predict(new_customer_scaled)[0]

cluster_profile = df[df['Cluster'] == predicted_cluster][['Age','Annual Income','Spending Score']].mean()
cluster_size    = (df['Cluster'] == predicted_cluster).sum()

print()
print('=' * 55)
print('        MEANSHIFT — PREDICTION RESULT')
print('=' * 55)
print(f'  Input  → Age: {input_age}, Gender: {input_gender}')
print(f'           Annual Income: {input_annual_income:,.0f}')
print(f'           Spending Score: {input_spending_score}')
print()
print(f'  ✅ Predicted Cluster : {predicted_cluster}')
print(f'  👥 Cluster Size      : {cluster_size} customers')
print()
print('  Cluster Average Profile:')
print(f'    Average Age          : {cluster_profile["Age"]:.1f}')
print(f'    Average Annual Income: {cluster_profile["Annual Income"]:,.0f}')
print(f'    Average Spending Score: {cluster_profile["Spending Score"]:.1f}')
print('=' * 55)

In [ ]:
new_pca = pca.transform(new_customer_scaled)

plt.figure(figsize=(9, 6))
palette = sns.color_palette('tab10', n_clusters)
for c in range(n_clusters):
    mask = cluster_labels == c
    lbl = f'Cluster {c} ← YOU' if c == predicted_cluster else f'Cluster {c}'
    alpha = 0.6 if c == predicted_cluster else 0.25
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                color=palette[c], label=lbl, alpha=alpha, s=15)

plt.scatter(new_pca[0, 0], new_pca[0, 1],
            marker='*', s=500, color='black', zorder=10, label='New Customer')

plt.title(f'MeanShift — New Customer falls in Cluster {predicted_cluster}', fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.savefig('meanshift_prediction.png', dpi=150)
plt.show()